# SaulLM-7B-Instruct-v1 — local inference

Runs Equall's legal LLM locally via Ollama (Q4_K_M quantization, ~4.4 GB).

**Prereqs** (one-time): `brew install ollama` and `ollama pull hf.co/mradermacher/Saul-7B-Instruct-v1-GGUF:Q4_K_M` (already done on this machine; the cells below verify).

In [1]:
import json, os, shutil, subprocess, time
import requests

OLLAMA_URL = "http://localhost:11434"
MODEL = "saul-7b"

assert shutil.which("ollama"), "ollama not installed — run: brew install ollama"

def server_up():
    try:
        requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        return True
    except requests.exceptions.RequestException:
        return False

if not server_up():
    env = {**os.environ, "OLLAMA_FLASH_ATTENTION": "1", "OLLAMA_KV_CACHE_TYPE": "q8_0"}
    subprocess.Popen(["ollama", "serve"], env=env,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if server_up(): break
        time.sleep(1)

tags = requests.get(f"{OLLAMA_URL}/api/tags").json()
names = [m["name"] for m in tags["models"]]
assert any(n.startswith(MODEL) for n in names), f"{MODEL} not pulled. Models present: {names}"
print("ollama up, model available:", MODEL)

ollama up, model available: saul-7b


In [2]:
class SaulChat:
    def __init__(self, model=MODEL, system=None, **options):
        self.model = model
        self.options = options
        self.history = [{"role": "system", "content": system}] if system else []

    def ask(self, prompt):
        self.history.append({"role": "user", "content": prompt})
        r = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json={"model": self.model, "messages": self.history,
                  "stream": False, "options": self.options},
            timeout=600,
        ).json()
        msg = r["message"]
        self.history.append(msg)
        return msg["content"]

    def reset(self, system=None):
        self.history = [{"role": "system", "content": system}] if system else []

def generate(prompt, model=MODEL, **options):
    r = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False, "options": options},
        timeout=600,
    ).json()
    return r["response"]

## Single-turn

In [3]:
print(generate("State the holding in Marbury v. Madison in one sentence.", temperature=0.2))

 In Marbury v. Madison, the Supreme Court established the principle of judicial review, asserting that the federal courts have the power to strike down laws and actions by the executive branch if they are deemed unconstitutional.


## Multi-turn

In [4]:
chat = SaulChat(system="You are a legal research assistant. Answer concisely and cite the doctrinal source where relevant.",
                temperature=0.3)
print(chat.ask("Explain the doctrine of res ipsa loquitur."))

 Res ipsa loquitur is a Latin phrase that translates to "the thing speaks for itself." It is a legal doctrine used in tort law, particularly in negligence cases, to establish liability without direct evidence of fault. The doctrine allows the court to infer negligence based on the circumstances surrounding an accident or injury.

In order to apply res ipsa loquitur, three conditions must be met:

1. The accident or injury must have been caused by an instrumentality within the exclusive control of the defendant. This means that the object or device responsible for causing the harm was under the defendant's sole possession and management at the time of the incident.

2. The accident or injury must not have occurred as a result of any voluntary action or contribution on the part of the plaintiff. In other words, the plaintiff cannot be deemed to have contributed to their own harm.

3. The accident or injury must be of a kind that does not ordinarily occur in the absence of someone's negli

In [5]:
print(chat.ask("Now contrast it with negligence per se."))

 Negligence per se is another legal doctrine in tort law that deals with negligence. Unlike res ipsa loquitur, which infers negligence based on the circumstances surrounding an accident or injury, negligence per se establishes liability without requiring a showing of causation.

Negligence per se occurs when there is a violation of a statute or regulation that was enacted to protect the public from harm. In such cases, the defendant's actions are considered negligent as a matter of law because they have violated a duty imposed by the statute or regulation. The plaintiff need only prove that the defendant violated the relevant law and that this violation caused their injury.

In contrast to res ipsa loquitur, which shifts the burden of proof from the plaintiff to the defendant, negligence per se does not shift the burden of proof. Instead, it provides a shortcut for establishing liability by treating the violation itself as evidence of negligence.


## Useful `options` (passed to Ollama)
- `temperature` — sampling temperature (default 0.8)
- `top_p`, `top_k` — nucleus / top-k sampling
- `num_predict` — max tokens to generate (-1 = unlimited)
- `num_ctx` — context window (Saul-7B supports 4096)
- `seed` — for reproducibility

Full list: https://github.com/ollama/ollama/blob/main/docs/modelfile.md#parameter